<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Practical: Decision Trees and Random Forests Project

*Session 5 · Notebook 03.05 · Practical · Student version*

## About this notebook

This is a hands-on **project lab** for tree-based classifiers. You met a single decision tree and a random forest in the lecture (notebook 03.04); here you run the full workflow yourself on a real credit dataset: explore it, encode its categorical columns, train a single tree, train a forest, and compare them honestly.

Along the way you will hit a very common real-world problem, **class imbalance**, and see why accuracy alone can be dangerously misleading.

**scikit-learn documentation for the model(s) used in this notebook:** [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) and [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

## About the exercises

Each task appears as two cells: a **question** (markdown, sometimes with a `> *Hint:*`) and an empty **`# Your turn`** cell for you to write your answer. Work top to bottom, since later tasks reuse variables you create earlier; your coach has the worked solutions.

## About the data

`german_credit.csv` is the classic **German Credit** dataset: 1,000 past loan applicants, each described by 20 features (a mix of categorical and numeric), and a target column **`risk`** labelling each as `good` or `bad`.

- **Categorical** features include `checking_status`, `credit_history`, `purpose`, `savings_status`, `employment_since`, `housing`, `job`, and more.
- **Numeric** features include `duration_months`, `credit_amount`, `age_years`, `installment_rate`, and others.
- The target is **imbalanced**: 700 `good` and 300 `bad`. Only 30 percent of applicants are the class we most want to catch.

We will treat **`bad` as the positive class (1)**: in risk work, the event you are trying to detect (a loan that goes bad) is the "positive" outcome for the model, even though it is a negative outcome for the business.

## Why this matters for risk analysis

Tree-based models are a mainstay of credit and fraud modelling for two reasons:

- **A single decision tree is a set of explainable if-then rules.** "If checking balance is negative AND duration > 30 months, then high risk." That transparency is valuable when you must justify a decision to a regulator or a declined customer. The cost is that one deep tree **overfits**: it memorises the training data and generalises poorly.
- **A random forest fixes that by averaging many varied trees.** Each tree sees a random subset of rows and features, so their individual mistakes cancel out. You trade some direct explainability for a big gain in stability and accuracy, and you still get **feature importances** telling you which drivers matter most.

The deeper lesson in this lab is about **imbalance**. With 70 percent good loans, a model that blindly predicts "good" for everyone scores 70 percent accuracy while catching **zero** bad loans, which is worthless. That is why we read precision and recall on the `bad` class, not headline accuracy.

## Index

1. [Setup](#setup)
2. [Get the data](#data)
3. [Exploratory data analysis](#eda)
4. [Set up the data: encode categoricals](#setup-data)
5. [Train / test split](#split)
6. [Train a single decision tree](#tree)
7. [Train a random forest](#forest)
8. [Compare the models and read the feature importances](#compare)
9. [Further practice](#further)
10. [Key takeaways](#takeaways)

<a id="setup"></a>
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

sns.set_style("whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Setup complete.")

<a id="data"></a>
## 2. Get the data

### Exercise 1: Load the data

Read `../../datasets/german_credit.csv` into a DataFrame called `loans`. Show `loans.head()`.

In [ ]:
# Your turn


### Exercise 2: Inspect the data

Call `loans.info()` to see the columns and dtypes (note which are `object` = categorical), and `loans.describe()` for the numeric columns. Then print the value counts of the target `risk`.

In [ ]:
# Your turn


<a id="eda"></a>
## 3. Exploratory data analysis

Before modelling, look at how a few features relate to risk. Do not worry about making the plots pretty; the point is to build intuition.

### Exercise 3: Credit amount by risk class

Create a single figure with two overlaid histograms of `credit_amount`: one for `good` loans and one for `bad` loans. Give them different colours, a legend, and some transparency so both are visible.

> *Hint:* filter the DataFrame twice (`loans[loans["risk"]=="good"]`) and call `.hist()` or `plt.hist()` on the `credit_amount` column of each, passing `alpha=0.5` and a `label`.

In [ ]:
# Your turn


### Exercise 4: Loan purpose vs risk

Create a seaborn `countplot` of `purpose` with the colour hue set by `risk`. Rotate the x labels so they are readable. Which purposes have the worst mix of bad loans?

> *Hint:* `sns.countplot(data=loans, x="purpose", hue="risk")`, then `plt.xticks(rotation=45, ha="right")`.

In [ ]:
# Your turn


### Exercise 5: Duration vs credit amount

Draw a seaborn `jointplot` of `duration_months` (x) against `credit_amount` (y). Is there a trend? (Longer loans tend to be larger.)

> *Hint:* `sns.jointplot(data=loans, x="duration_months", y="credit_amount")`.

In [ ]:
# Your turn


<a id="setup-data"></a>
## 4. Set up the data: encode categoricals

Scikit-learn models only accept numbers, but many of our columns are text (`checking_status`, `purpose`, ...). We must convert them.

- For the **target**, we map `good -> 0` and `bad -> 1` so that "bad" (the risk event) is the positive class.
- For the **categorical predictors**, we use **one-hot encoding** via `pd.get_dummies(..., drop_first=True)`, which turns each category into its own 0/1 column. `drop_first=True` removes one redundant column per feature to avoid the "dummy variable trap".

### Exercise 6: Encode the target

Create a new column (or overwrite `risk`) so that `bad = 1` and `good = 0`. Confirm the mean equals the bad rate you saw earlier (about 0.30).

> *Hint:* `(loans["risk"] == "bad").astype(int)`.

In [ ]:
# Your turn


### Exercise 7: One-hot encode the categorical features

Find the categorical columns (the `object` dtype columns), then use `pd.get_dummies(loans, columns=cat_feats, drop_first=True)` to build a fully numeric DataFrame called `final_data`. Print its shape and confirm every column is now numeric.

> *Hint:* `cat_feats = loans.select_dtypes(include="object").columns.tolist()`.

In [ ]:
# Your turn


<a id="split"></a>
## 5. Train / test split

### Exercise 8: Split features and target, then train/test split

Create `X` (everything except `risk`) and `y` (`risk`). Split into train/test with `test_size=0.30`, `random_state=RANDOM_STATE`, and `stratify=y` so both sets keep the 30 percent bad rate.

> *Hint:* stratifying matters with imbalanced data, otherwise one split could end up with very few bad cases.

In [ ]:
# Your turn


<a id="tree"></a>
## 6. Train a single decision tree

We start with one decision tree so we have a baseline to beat.

### Exercise 9: Train a decision tree

Create a `DecisionTreeClassifier(random_state=RANDOM_STATE)` called `dtree` and fit it on the training data.

In [ ]:
# Your turn


### Exercise 10: Evaluate the decision tree

Predict on `X_test` (call it `dtree_pred`), then print the confusion matrix and the classification report. Pay attention to the **recall on class 1 (bad)**: of all the truly bad loans, how many did the tree catch?

In [ ]:
# Your turn


A single unrestricted tree grows very deep and memorises the training data. On the test set it typically catches only about half the bad loans, and its overall performance is unstable. Let us see if a forest does better.

<a id="forest"></a>
## 7. Train a random forest

### Exercise 11: Train a random forest

Create a `RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)` called `rfc` and fit it on the training data.

In [ ]:
# Your turn


### Exercise 12: Evaluate the random forest

Predict on `X_test` (call it `rfc_pred`), then print the classification report and confusion matrix. **Do you notice anything strange?** Compare the recall on the `bad` class to its recall on the `good` class.

In [ ]:
# Your turn


**The strange thing:** the forest's overall accuracy looks respectable, but its **recall on the `bad` class is poor**, often worse than the single tree's. The forest, chasing overall accuracy on an imbalanced dataset, learns that the safest bet is to call almost everyone `good`. It is right most of the time (70 percent are good) but it misses most of the bad loans, which are exactly the ones a lender cares about.

This is the central trap of imbalanced classification: **a model can look accurate while being useless for its actual purpose.** You only see it by reading per-class recall, never headline accuracy.

<a id="compare"></a>
## 8. Compare the models and read the feature importances

### Exercise 13: Which model performed better?

Print the test accuracy of both models side by side, and (more importantly) their **recall on the bad class**. Comment: which would you rather deploy, and why is accuracy the wrong headline number here?

> *Hint:* `from sklearn.metrics import accuracy_score, recall_score`. Use `recall_score(y_test, pred, pos_label=1)`.

In [ ]:
# Your turn


### Exercise 14: Feature importances

A forest tells you which features drove its decisions. Extract `rfc.feature_importances_`, put them in a Series indexed by `X.columns`, sort descending, and plot the top 10 as a horizontal bar chart. Which features matter most for predicting risk?

In [ ]:
# Your turn


<a id="further"></a>
## 9. Further practice

The headline finding, that both models under-detect bad loans, is the starting point for real work, not the end. Try:

1. **Move the decision threshold** (done in the exercise below). The most direct lever: the forest calls a loan `bad` only when its predicted probability clears 0.5. Lower that cut-off and you catch more bad loans, at the cost of more false alarms.
2. **Resample the training data.** Oversample the rare `bad` class (for example `imblearn`'s `RandomOverSampler`) or undersample the `good` class, then refit. (`class_weight="balanced"` is the equivalent idea but, for random forests specifically, it often barely moves recall, so resampling or thresholding tends to work better here.)
3. **Tune the forest.** Use `GridSearchCV` over `n_estimators`, `max_depth`, and `min_samples_leaf` with `scoring="recall"` or `"f1"` so tuning optimises for catching bad loans, not accuracy.

### Exercise 15 (stretch): move the decision threshold to catch more bad loans

By default the forest labels a loan `bad` only when its predicted probability exceeds **0.5**. Because bad loans are rare, few clear that bar, which is why recall on `bad` is low.

Get the predicted probabilities with `rfc.predict_proba(X_test)[:, 1]`, then relabel using a lower cut-off of **0.30**. Print the recall and precision on the `bad` class at 0.5 versus 0.30. How much does recall improve, and what does it cost?

> *Hint:* `(proba >= 0.30).astype(int)` gives the new predictions.

In [ ]:
# Your turn


<a id="takeaways"></a>
## 10. Key takeaways

| Step | What you did | Why it matters |
|---|---|---|
| Encode | `get_dummies(drop_first=True)`, map target | sklearn needs numbers; drop_first avoids the dummy trap |
| Stratified split | `stratify=y` | Keep the rare class proportion in both train and test |
| Single tree | `DecisionTreeClassifier` | Explainable rules, but one deep tree overfits |
| Random forest | `RandomForestClassifier` | Averaging many trees is more stable and gives feature importances |
| Read per-class recall | Not just accuracy | On imbalanced data, accuracy hides that bad loans go undetected |
| Adjust the threshold | `predict_proba` + lower cut-off | Directly trades precision for recall to catch more of the rare, costly class |

**The one-line lesson:** on imbalanced risk data, a model that looks accurate can still miss almost every case you care about. Judge tree models by recall on the event class, and move the decision threshold (or resample) when that recall is too low.

Great job. You have built, compared, and diagnosed two tree-based classifiers on a real credit dataset.